Libraries

In [17]:
pip install lasio

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import lasio
import pandas as pd
from pathlib import Path

In [17]:
def clean_path(p):
    """
    Fixes common macOS path issues:
    - Expands ~
    - Removes accidental quotes
    - Resolves absolute path
    """
    return Path(str(p).strip().strip('"').strip("'")).expanduser().resolve()

LAS Reader Function

In [9]:
def read_las_file_flexible(file_path):
    file_path = clean_path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"❌ File not found: {file_path}")
    
    try:
        # IMPORTANT: prevent lasio from auto-removing dummy values
        las = lasio.read(
            file_path,
            ignore_header_errors=True,
            null_policy="none"   # << THIS is the key fix
        )
    except Exception as e:
        raise ValueError(f"❌ File is not valid LAS content: {file_path}\n{e}")
    
    curve_names = [curve.mnemonic for curve in las.curves]
    curve_units = [curve.unit for curve in las.curves]
    
    # Now df still contains original -999.25 values
    data_df = las.df()
    
    # Read NULL value from header
    null_value = float(las.well["NULL"].value)
    
    return curve_names, curve_units, data_df, null_value


CSV Formatter Function

In [10]:
def build_csv_dataframe(curve_names, curve_units, data_df, null_value, remove_dummy):
    
    data_df = data_df.reset_index()
    
    if remove_dummy:
        data_df = data_df.replace(null_value, "")
    
    data_values = data_df.values.tolist()
    
    final_table = []
    final_table.append(curve_names)
    final_table.append(curve_units)
    final_table.extend(data_values)
    
    return pd.DataFrame(final_table)


Single LAS to CSV converter.

In [11]:
def convert_single_las_to_csv(file_path, output_directory=None):
    
    file_path = clean_path(file_path)
    
    if output_directory:
        output_directory = Path(output_directory)
        output_directory.mkdir(parents=True, exist_ok=True)
        output_csv_path = output_directory / (file_path.stem + ".csv")
    else:
        output_csv_path = file_path.with_suffix(".csv")
    
    print(f"Processing: {file_path.name}")
    
    curve_names, curve_units, data_df, null_value = read_las_file_flexible(file_path)
    
    final_df = build_csv_dataframe(
        curve_names,
        curve_units,
        data_df,
        null_value=null_value,
        remove_dummy=REMOVE_DUMMY_VALUES   # << uses your switch
    )
    
    final_df.to_csv(output_csv_path, index=False, header=False)
    
    print(f"✅ Saved CSV → {output_csv_path}")
    
    return output_csv_path


Batch LAS to CSV converter

In [12]:
def convert_batch_las_to_csv(parent_directory, output_directory=None):
    """
    Batch converts LAS content files.
    Accepts:
      - .las files
      - .txt files (renamed LAS)
    Ignores all other files.
    """
    
    parent_directory = Path(parent_directory)
    
    # Collect only .las and .txt files
    las_files = list(parent_directory.glob("*.las"))
    txt_files = list(parent_directory.glob("*.txt"))
    
    files = las_files + txt_files
    
    if not files:
        print("❌ No LAS or TXT files found!")
        return
    
    print(f"Found {len(files)} LAS/TXT file(s)\n")
    
    success = 0
    
    for i, file in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {file.name}")
        try:
            convert_single_las_to_csv(file, output_directory)
            success += 1
        except Exception as e:
            print("⚠️ Skipped — not valid LAS content\n")
    
    print(f"🎉 Completed! {success} file(s) converted successfully.")


Dummy Value Control Switch (NEW CELL)

In [13]:
# ================================
# Dummy Value Handling Switch
# ================================

# True  → Replace dummy values with empty cells in CSV
# False → Keep original dummy values in CSV

REMOVE_DUMMY_VALUES = True


Single file execution

In [18]:
# Set LAS file path
las_path = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/las/drill_mech_depth_15aug20.las"

# Convert single LAS to CSV
convert_single_las_to_csv(las_path)


Only engine='normal' can read wrapped files


Processing: drill_mech_depth_15aug20.las
✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/las/drill_mech_depth_15aug20.csv


PosixPath('/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/las/drill_mech_depth_15aug20.csv')

Batch execution

In [ ]:
# Set parent folder containing multiple LAS / TXT files
parent_folder = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/las"

# (Optional) Set output folder for CSV files
output_folder = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv"

# Convert all LAS/TXT files in folder to CSV
convert_batch_las_to_csv(parent_folder, output_folder)


Found 27 LAS/TXT file(s)

[1/27] snl_time_02_07oct20.las
Processing: snl_time_02_07oct20.las


Only engine='normal' can read wrapped files
Only engine='normal' can read wrapped files
Only engine='normal' can read wrapped files


✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_time_02_07oct20.csv
[2/27] snl_ccl_09_07oct20.las
Processing: snl_ccl_09_07oct20.las
✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_ccl_09_07oct20.csv
[3/27] snl_ccl_10_07oct20.las
Processing: snl_ccl_10_07oct20.las


Only engine='normal' can read wrapped files


✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_ccl_10_07oct20.csv
[4/27] snl_ccl_05_07oct20.las
Processing: snl_ccl_05_07oct20.las
✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_ccl_05_07oct20.csv
[5/27] snl_02_07oct20.las
Processing: snl_02_07oct20.las


...

Only engine='normal' can read wrapped files
Only engine='normal' can read wrapped files


✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_time_03_07oct20.csv
[22/27] snl_ccl_02_06oct20.las
Processing: snl_ccl_02_06oct20.las


Only engine='normal' can read wrapped files


✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_ccl_02_06oct20.csv
[23/27] snl_ccl_02_07oct20.las
Processing: snl_ccl_02_07oct20.las
✅ Saved CSV → /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/snl_ccl_02_07oct20.csv
[24/27] drill_mech_time_03aug20.las
Processing: drill_mech_time_03aug20.las


Only engine='normal' can read wrapped files
